# Comment Letter Evaluation with OpenAI

This notebook loads comment files (JSON or CSV) one at a time and asks the OpenAI API to score each comment on relevance, reasoning, evidence, impact, actionability, and structure/formatting.

Supported input formats:
- **Individual JSON files** — the original per-comment files in `comments/`
- **Statt CSV files** — columns: `request_id`, `display_name`, `entity_type`, `stance`, `status_code`, `comment_text` (where `comment_text` is a JSON string with a `draft` key)
- **Gemini CSV files** — columns: `Display Name` / `display_name`, `Organization`, `Stance`, `Comment Letter` / `comment_letter` (plain text)

Before running, set `OPENAI_API_KEY` in your environment. Results are saved to JSONL and CSV in the `LLM-eval` folder.

In [8]:
# Install dependencies (run once)
#%pip install -q openai jsonschema pandas tqdm python-dotenv

## Configure OpenAI API Client

This section loads the API key from the environment and sets default request settings.

In [9]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is not set in the environment.")

client = OpenAI(api_key=api_key)

MODEL = "gpt-5.4-mini"
TEMPERATURE = 0.2
MAX_TOKENS = 1000

## Define Evaluation Criteria and Scoring Schema

Scores are 1 to 5 for each criterion, with short justifications. The schema enforces a strict JSON response.

In [10]:
from jsonschema import validate, Draft7Validator

RUBRIC = {
    "relevance": "How directly the comment addresses the docket topic and requested issues.",
    "reasoning": "Logical coherence and clarity of arguments.",
    "evidence": "Use of facts, data, citations, or concrete examples.",
    "impact": "Specificity and plausibility of claimed impacts.",
    "actionability": "Provides clear, feasible recommendations or requests.",
    "structure_formatting": "Organization, readability, and professional formatting.",
    "overall": "Holistic quality across all criteria."
}

RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "scores": {
            "type": "object",
            "properties": {
                "relevance": {"type": "integer", "minimum": 1, "maximum": 5},
                "reasoning": {"type": "integer", "minimum": 1, "maximum": 5},
                "evidence": {"type": "integer", "minimum": 1, "maximum": 5},
                "impact": {"type": "integer", "minimum": 1, "maximum": 5},
                "actionability": {"type": "integer", "minimum": 1, "maximum": 5},
                "structure_formatting": {"type": "integer", "minimum": 1, "maximum": 5},
                "overall": {"type": "integer", "minimum": 1, "maximum": 5}
            },
            "required": [
                "relevance",
                "reasoning",
                "evidence",
                "impact",
                "actionability",
                "structure_formatting",
                "overall"
            ]
        },
        "rationales": {
            "type": "object",
            "properties": {
                "relevance": {"type": "string"},
                "reasoning": {"type": "string"},
                "evidence": {"type": "string"},
                "impact": {"type": "string"},
                "actionability": {"type": "string"},
                "structure_formatting": {"type": "string"},
                "overall": {"type": "string"}
            },
            "required": [
                "relevance",
                "reasoning",
                "evidence",
                "impact",
                "actionability",
                "structure_formatting",
                "overall"
            ]
        },
        "overall_summary": {"type": "string"}
    },
    "required": ["scores", "rationales", "overall_summary"]
}

_validator = Draft7Validator(RESPONSE_SCHEMA)


def validate_response(data):
    errors = sorted(_validator.iter_errors(data), key=lambda e: e.path)
    if errors:
        raise ValueError("Schema validation failed: " + "; ".join([e.message for e in errors]))
    return data

## Load Comment Files from LLM-eval

Set `COMMENTS_DIR` to the folder containing comment files. Both individual JSON files and CSV files (Statt and Gemini formats) are supported.

CSV files are detected automatically by their columns:
- **Statt format**: has a `comment_text` column whose value is a JSON string with a `draft` key
- **Gemini format**: has a `Comment Letter` or `comment_letter` column with plain text

In [11]:
import csv
import html
import json
import re
from pathlib import Path

BASE_DIR = Path.cwd()
COMMENTS_DIR = BASE_DIR / "comments"
CONTEXTS_PATH = BASE_DIR / "policy_contexts.json"


# ── JSON helpers (unchanged) ──────────────────────────────────────────────

def list_json_files(folder: Path):
    return sorted([
        p for p in folder.rglob("*.json")
        if p.is_file() and not p.name.endswith("_personas.json")
    ])


def normalize_text(text):
    text = html.unescape(text)
    text = text.replace("<br/>", "\n").replace("<br>", "\n").replace("<br />", "\n")
    return text.strip()


def extract_comment_text(payload):
    """Extract comment text from a JSON payload (individual comment files)."""
    text_obj = payload.get("text")
    if isinstance(text_obj, dict):
        full_text = text_obj.get("fullText")
        if isinstance(full_text, str) and full_text.strip():
            return normalize_text(full_text)

    candidate_keys = ["comment", "comment_text", "commentText", "text", "content", "body"]
    for key in candidate_keys:
        value = payload.get(key)
        if isinstance(value, str) and value.strip():
            return normalize_text(value)

    text_parts = [v for v in payload.values() if isinstance(v, str) and v.strip()]
    if text_parts:
        return normalize_text("\n\n".join(text_parts))
    return None


def extract_policy_id(payload, comment_text, path: Path):
    for key in ["docketId", "docket_id", "docketID", "docket", "policy_id", "policyId"]:
        value = payload.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()

    if isinstance(comment_text, str):
        patterns = [
            r"\bEPA-[A-Z]{2}-[A-Z]{2}-\d{4}-\d{4}\b",
            r"\bFDA-\d{4}-[A-Z]-\d{4}\b",
            r"\bFMCSA-\d{4}-\d{4}\b",
            r"\b[A-Z]{3,}-\d{4}-[A-Z]-\d{3,5}\b",
            r"\b[A-Z]{3,}-\d{4}-\d{3,6}\b",
        ]
        for pattern in patterns:
            match = re.search(pattern, comment_text)
            if match:
                return match.group(0)

    return "unknown"


def extract_source_label(payload, path: Path):
    for key in ["source_label", "author_type", "label"]:
        value = payload.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip().lower()
    for part in path.parts:
        if part.lower() in {"human", "ai"}:
            return part.lower()
    return "unknown"


# ── CSV helpers ───────────────────────────────────────────────────────────

def list_csv_files(folder: Path):
    """Return all CSV files in the comments folder."""
    return sorted([p for p in folder.rglob("*.csv") if p.is_file()])


def _infer_source_label_from_filename(path: Path) -> str:
    """Infer 'statt' or 'gemini' from the CSV filename, else 'unknown'."""
    name_lower = path.name.lower()
    if "statt" in name_lower:
        return "statt"
    if "gemini" in name_lower:
        return "gemini"
    return "unknown"


def _infer_policy_prefix_from_filename(path: Path) -> str:
    """Infer a rough policy prefix (e.g. 'epa', 'fda', 'fmcsa') from filename."""
    name_lower = path.stem.lower()
    for prefix in ["epa", "fda", "fmcsa"]:
        if name_lower.startswith(prefix):
            return prefix.upper()
    return "unknown"


def _extract_comment_text_from_csv_row(row: dict) -> str | None:
    """
    Extract the plain comment text from a CSV row.

    Handles:
    - Statt format: 'comment_text' column whose value is JSON with a 'draft' key
    - Gemini format: 'Comment Letter' or 'comment_letter' column with plain text
    - Generic fallback: any column whose name contains 'comment' or 'letter'
    """
    # ---- Statt: comment_text is JSON-encoded with a 'draft' key ----
    raw = row.get("comment_text", "")
    if raw and raw.strip().startswith("{"):
        try:
            parsed = json.loads(raw)
            draft = parsed.get("draft")
            if isinstance(draft, str) and draft.strip():
                return normalize_text(draft)
        except json.JSONDecodeError:
            pass
    if raw and raw.strip():
        return normalize_text(raw)

    # ---- Gemini: 'Comment Letter' or 'comment_letter' ----
    for key in ["Comment Letter", "comment_letter"]:
        val = row.get(key, "")
        if val and val.strip():
            return normalize_text(val)

    # ---- Generic fallback: any column whose name suggests comment text ----
    for col, val in row.items():
        if any(kw in col.lower() for kw in ["comment", "letter", "text", "body"]):
            if isinstance(val, str) and val.strip():
                return normalize_text(val)

    return None


def _extract_display_name_from_csv_row(row: dict) -> str:
    """Try common column names for a commenter display name."""
    for key in ["display_name", "Display Name", "name", "Name", "organization", "Organization"]:
        val = row.get(key, "")
        if val and val.strip():
            return val.strip()
    return ""


def load_csv_records(csv_path: Path) -> list[dict]:
    """
    Load a CSV file and return a list of normalised comment records, each with:
      - 'source'        : unique identifier string for the record
      - 'comment_text'  : extracted plain text (or None)
      - 'policy_id'     : docket ID inferred from text or filename
      - 'source_label'  : 'statt', 'gemini', or 'unknown'
      - 'display_name'  : commenter name / organisation
      - 'raw_row'       : the original CSV row dict (for debugging)
    """
    source_label = _infer_source_label_from_filename(csv_path)
    policy_prefix = _infer_policy_prefix_from_filename(csv_path)
    records = []

    with open(csv_path, newline="", encoding="utf-8") as fh:
        reader = csv.DictReader(fh)
        for idx, row in enumerate(reader):
            comment_text = _extract_comment_text_from_csv_row(row)
            display_name = _extract_display_name_from_csv_row(row)

            # Derive a unique source string
            request_id = row.get("request_id", "").strip()
            source = (
                f"csv:{csv_path.name}:{request_id}"
                if request_id
                else f"csv:{csv_path.name}:row{idx}"
            )

            # Try to find a docket ID in the text, fall back to file-based prefix
            policy_id = "unknown"
            if isinstance(comment_text, str):
                patterns = [
                    r"\bEPA-[A-Z]{2}-[A-Z]{2}-\d{4}-\d{4}\b",
                    r"\bFDA-\d{4}-[A-Z]-\d{4}\b",
                    r"\bFMCSA-\d{4}-\d{4}\b",
                    r"\b[A-Z]{3,}-\d{4}-[A-Z]-\d{3,5}\b",
                    r"\b[A-Z]{3,}-\d{4}-\d{3,6}\b",
                ]
                for pattern in patterns:
                    match = re.search(pattern, comment_text)
                    if match:
                        policy_id = match.group(0)
                        break
            if policy_id == "unknown" and policy_prefix != "unknown":
                policy_id = policy_prefix

            records.append({
                "source": source,
                "comment_text": comment_text,
                "policy_id": policy_id,
                "source_label": source_label,
                "display_name": display_name,
                "raw_row": dict(row),
            })

    return records


# ── Discover files ────────────────────────────────────────────────────────

def load_policy_contexts(path: Path):
    if not path.exists():
        return {}
    data = json.loads(path.read_text(encoding="utf-8"))
    return data if isinstance(data, dict) else {}


policy_contexts = load_policy_contexts(CONTEXTS_PATH)
json_files = list_json_files(COMMENTS_DIR)
csv_files = list_csv_files(COMMENTS_DIR)

# Pre-load all CSV records into a flat list
csv_records = []
for csv_path in csv_files:
    try:
        csv_records.extend(load_csv_records(csv_path))
    except Exception as exc:
        print(f"Warning: could not load {csv_path.name}: {exc}")

print(f"Found {len(json_files)} JSON file(s) and {len(csv_records)} CSV row(s) across {len(csv_files)} CSV file(s).")


Found 300 JSON file(s) and 634 CSV row(s) across 6 CSV file(s).


## Build Prompt and Call the OpenAI API

The prompt requests a JSON-only response that follows the schema.

In [12]:
import textwrap


def build_prompt(comment_text, policy_id=None, policy_context=None):
    rubric_lines = "\n".join([f"- {k}: {v}" for k, v in RUBRIC.items()])
    policy_block = ""
    if policy_id or policy_context:
        policy_block = textwrap.dedent(
            f"""
            Policy context:
            - Policy ID: {policy_id or "unknown"}
            - Summary: {policy_context or "(no summary provided)"}
            """
        )

    return textwrap.dedent(
        f"""
        You are evaluating a public comment letter. Score each criterion from 1 (poor) to 5 (excellent).
        Provide a short rationale per criterion and a brief overall summary.

        Criteria:
        {rubric_lines}

        {policy_block}
        Return ONLY valid JSON with this structure:
        {{
          "scores": {{"relevance": 1-5, "reasoning": 1-5, "evidence": 1-5, "impact": 1-5,
                      "actionability": 1-5, "structure_formatting": 1-5, "overall": 1-5}},
          "rationales": {{"relevance": "...", "reasoning": "...", "evidence": "...", "impact": "...",
                         "actionability": "...", "structure_formatting": "...", "overall": "..."}},
          "overall_summary": "..."
        }}

        Comment:
        """ + comment_text.strip()
    )


def call_openai(comment_text, policy_id=None, policy_context=None):
    prompt = build_prompt(comment_text, policy_id=policy_id, policy_context=policy_context)
    response = client.responses.create(
        model=MODEL,
        input=prompt,
        temperature=TEMPERATURE,
        max_output_tokens=MAX_TOKENS,
    )
    return response.output_text

## Parse Model Response and Validate JSON Output

This attempts to extract JSON if the model returns extra text, then validates the schema.

In [13]:
import re


def extract_json(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        raise ValueError("No JSON object found in response")
    return json.loads(match.group(0))


def evaluate_comment(comment_text, policy_id=None, policy_context=None):
    raw = call_openai(comment_text, policy_id=policy_id, policy_context=policy_context)
    data = extract_json(raw)
    validate_response(data)
    return data

## Process Files One-by-One and Save Results

This runs through files, collects results, and writes JSONL and CSV outputs.

In [14]:
import pandas as pd
from tqdm import tqdm

OUTPUT_JSONL = BASE_DIR / "comment_scores.jsonl"
OUTPUT_CSV   = BASE_DIR / "comment_scores.csv"

MAX_FILES = None  # set to an integer for a smaller test run
RESUME = True     # skip already-scored entries in OUTPUT_JSONL

results = []
processed = set()

if RESUME and OUTPUT_JSONL.exists():
    for line in OUTPUT_JSONL.read_text(encoding="utf-8").splitlines():
        try:
            row = json.loads(line)
            if isinstance(row, dict) and "file" in row:
                results.append(row)
                processed.add(row["file"])
        except json.JSONDecodeError:
            continue


# ── Build a unified list of work items ───────────────────────────────────
# Each item is a dict with keys: source, comment_text, policy_id, source_label
# (and optionally display_name).  JSON files are resolved lazily below.

work_items = []

# JSON files → lazy-load in the loop
for path in json_files:
    work_items.append({"_type": "json", "_path": path})

# CSV records → already loaded
for rec in csv_records:
    work_items.append({"_type": "csv", **rec})

if MAX_FILES is not None:
    work_items = work_items[:MAX_FILES]


# ── Process ───────────────────────────────────────────────────────────────

for item in tqdm(work_items):
    try:
        if item["_type"] == "json":
            path = item["_path"]
            source_key = str(path)
            if RESUME and source_key in processed:
                continue
            with path.open("r", encoding="utf-8") as f:
                payload = json.load(f)
            comment_text = extract_comment_text(payload)
            source_label = extract_source_label(payload, path)
            policy_id    = extract_policy_id(payload, comment_text, path)
            display_name = (
                payload.get("display_name")
                or payload.get("organizationName")
                or ""
            )
        else:  # CSV record
            source_key   = item["source"]
            if RESUME and source_key in processed:
                continue
            comment_text = item["comment_text"]
            source_label = item["source_label"]
            policy_id    = item["policy_id"]
            display_name = item.get("display_name", "")

        policy_context = policy_contexts.get(policy_id, "")

        if not comment_text:
            results.append({
                "file": source_key,
                "policy_id": policy_id,
                "source_label": source_label,
                "display_name": display_name,
                "error": "No comment text found",
            })
            continue

        evaluation = evaluate_comment(
            comment_text,
            policy_id=policy_id,
            policy_context=policy_context,
        )
        results.append({
            "file": source_key,
            "policy_id": policy_id,
            "source_label": source_label,
            "display_name": display_name,
            "comment_text": comment_text,
            **evaluation,
        })

    except Exception as exc:
        results.append({
            "file": item.get("source", str(item.get("_path", "unknown"))),
            "policy_id": "unknown",
            "source_label": "unknown",
            "display_name": "",
            "error": str(exc),
        })


# ── Write JSONL ───────────────────────────────────────────────────────────

with OUTPUT_JSONL.open("w", encoding="utf-8") as f:
    for row in results:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


# ── Write CSV ─────────────────────────────────────────────────────────────

def sanitize_csv_field(value):
    if value is None:
        return ""
    if not isinstance(value, str):
        return value
    cleaned = value.replace("\r\n", " ").replace("\n", " ").replace("\r", " ")
    return " ".join(cleaned.split())


flat_rows = []
for row in results:
    if "scores" in row:
        flat = {
            "file":          row["file"],
            "policy_id":     row.get("policy_id", "unknown"),
            "source_label":  row.get("source_label", "unknown"),
            "display_name":  row.get("display_name", ""),
            "comment_text":  sanitize_csv_field(row.get("comment_text", "")),
            "overall_summary": sanitize_csv_field(row.get("overall_summary", "")),
        }
        for k, v in row["scores"].items():
            flat[f"score_{k}"] = v
        for k, v in row["rationales"].items():
            flat[f"rationale_{k}"] = sanitize_csv_field(v)
        flat_rows.append(flat)
    else:
        flat_rows.append({
            "file":         row["file"],
            "policy_id":    row.get("policy_id", "unknown"),
            "source_label": row.get("source_label", "unknown"),
            "display_name": row.get("display_name", ""),
            "error":        sanitize_csv_field(row.get("error", "Unknown error")),
        })

pd.DataFrame(flat_rows).to_csv(OUTPUT_CSV, index=False)

OUTPUT_JSONL, OUTPUT_CSV


100%|██████████| 934/934 [23:15<00:00,  1.49s/it]  


(PosixPath('/Users/norazhan/Desktop/MDI/Statt/LLM-eval/comment_scores.jsonl'),
 PosixPath('/Users/norazhan/Desktop/MDI/Statt/LLM-eval/comment_scores.csv'))